## Лабораторная работа №7
### Fine-tuning DistilBERT на IMDB

Пайплайн разбит на две части: `Lab7_transformer_train.ipynb` — fine-tuning в Colab (T4 GPU), `Lab7_transformer_infer.ipynb` — метрики, PR-кривая, ROC-AUC, инференс.

## Что делаем

**Задача:** бинарная классификация отзывов IMDB (neg/pos).
**База:** DistilBERT (`distilbert-base-uncased`) — 6 трансформер-слоёв, hidden=768, 12 голов attention, ~66M параметров. Дистиллят BERT, ~97% качества при 2× скорости.
**Голова:** `Linear(768→768) + ReLU + Dropout + Linear(768→2)` (стандартная DistilBERT-голова).
**Fine-tuning:** 2 эпохи, AdamW `lr=2e-5`, `weight_decay=0.01`, batch=16, fp16.
**Подвыборка:** 4000 train / 2000 test (для скорости в Colab).
**Итог:** accuracy ≈ 0.89, ROC-AUC ≈ 0.95.

### Кривые обучения

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open("history.json") as f:
    log = json.load(f)

train_steps, train_loss = [], []
eval_epochs, eval_acc, eval_loss = [], [], []

for rec in log:
    if "loss" in rec and "eval_loss" not in rec:
        train_steps.append(rec["step"])
        train_loss.append(rec["loss"])
    if "eval_accuracy" in rec:
        eval_epochs.append(rec["epoch"])
        eval_acc.append(rec["eval_accuracy"])
        eval_loss.append(rec["eval_loss"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(train_steps, train_loss, label="train")
if eval_loss:
    eval_steps = [train_steps[-1] * (e / max(eval_epochs)) for e in eval_epochs]
    ax[0].plot(eval_steps, eval_loss, "o-", label="eval")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].set_title("loss")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(eval_epochs, eval_acc, "o-", color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy"); ax[1].set_title("eval accuracy")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

if eval_acc:
    print(f"final eval acc: {eval_acc[-1]:.4f}")
